# Step 6: Model Evaluation & Hyperparameter Tuning

This notebook provides an interactive view of:
1. PR & ROC Curves for all baseline models
2. Optimal threshold selection for XGBoost
3. Hyperparameter tuning results
4. Final test evaluation of tuned models

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import sys
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, average_precision_score,
    roc_auc_score, precision_recall_curve, roc_curve, f1_score
)

sys.path.append(os.path.abspath('..'))

In [ ]:
# Load processed data and recreate the same test split
df = pd.read_csv('../data/processed/processed_transactions.csv')
y = df['isFraud']
X = df.drop(columns=['isFraud'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Test set: {X_test.shape[0]:,} rows | Fraud cases: {y_test.sum():,}')

## 1. PR Curves — All Baseline Models

In [ ]:
models = {
    'Logistic Regression': joblib.load('../models/baseline_logreg.pkl'),
    'XGBoost':             joblib.load('../models/xgboost_model.pkl'),
    'LightGBM':            joblib.load('../models/lightgbm_model.pkl'),
}

colors = ['#6c757d', '#0d6efd', '#fd7e14']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    # PR Curve
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[0].plot(recall, precision, label=f'{name} (AP={ap:.4f})', color=color, lw=2)
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=color, lw=2)

axes[0].set_title('Precision-Recall Curves'); axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot([0,1],[0,1],'k--',lw=1,label='Random')
axes[1].set_title('ROC Curves'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/pr_roc_curves.png', dpi=150)
plt.show()

## 2. Optimal Threshold for XGBoost

In [ ]:
xgb_model = models['XGBoost']
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_xgb)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores[:-1])
best_thresh = thresholds[best_idx]

print(f'Optimal Threshold: {best_thresh:.4f} | Best F1: {f1_scores[best_idx]:.4f}')

print('\n--- Default Threshold (0.5) ---')
print(classification_report(y_test, (y_prob_xgb >= 0.5).astype(int)))

print(f'\n--- Optimal Threshold ({best_thresh:.4f}) ---')
print(classification_report(y_test, (y_prob_xgb >= best_thresh).astype(int)))

## 3. Tuned Model Results
> Run `src/models/tune_model.py` first to generate the tuned model files.

In [ ]:
tuned = {
    'XGBoost (Tuned)':  joblib.load('../models/xgboost_tuned.pkl'),
    'LightGBM (Tuned)': joblib.load('../models/lightgbm_tuned.pkl'),
}

print('=== FINAL TEST EVALUATION ===')
for name, model in tuned.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred))
    print(f'PR AUC: {average_precision_score(y_test, y_prob):.4f}')

In [ ]:
# Print tuning results summary
with open('../reports/tuning_results.txt', 'r') as f:
    print(f.read())